---

# 🎓 Self-Assignment: Build Your Own RAG System

## Mini Project — "Ask My Documents"

**Objective:** Build a complete, end-to-end RAG application using everything you learned in this course. You will create a system that can answer questions about **your own documents** (PDFs, text files, or web pages).

**Estimated Time:** 2–3 hours

**Difficulty:** ⭐⭐⭐ Intermediate

---

### 📋 Project Brief

> *You are an AI engineer at a company. Your team has a collection of internal documents (policies, technical guides, meeting notes). Build a RAG-powered Q&A system that lets employees ask natural language questions and get accurate, grounded answers from these documents.*

---

### ✅ Requirements

Your submission must include a working Jupyter notebook with the following **7 tasks**:

| Task | Description | Points |
|------|-------------|--------|
| **Task 1** | Load at least **3 documents** from at least **2 different sources** (e.g., PDF + Web, or PDF + TXT) | 10 |
| **Task 2** | Implement a chunking strategy with a **justified choice** of `chunk_size` and `chunk_overlap` — write a comment explaining your reasoning | 10 |
| **Task 3** | Store embeddings in a **Chroma** vector store with persistence enabled | 10 |
| **Task 4** | Build a **retriever** and demonstrate it works by showing top-K results for 3 different queries | 10 |
| **Task 5** | Write a **custom RAG prompt** that includes a system message with specific instructions (e.g., "answer in bullet points", "cite the source page", or "say I don't know if unsure") | 15 |
| **Task 6** | Assemble the **complete RAG chain** using LCEL (pipe operators) and test it with at least **5 questions** | 15 |
| **Task 7** | **Bonus Challenges** (pick at least one) — see below | 30 |

**Total: 100 points**

---

### 🌟 Task 7 — Bonus Challenges (pick at least one for full marks)

| Challenge | Description | Points |
|-----------|-------------|--------|
| **A. Multi-turn memory** | Add conversation history so follow-up questions work (e.g., "What about its pricing?" after asking about a product) | 10 |
| **B. Source citation** | Modify the chain to return **which documents** were used to answer each question (page number, filename) | 10 |
| **C. Evaluation harness** | Create 5 question-answer pairs as ground truth, then run your RAG chain and compare its answers against the ground truth (simple string match or LLM-as-judge) | 10 |
| **D. Chunking experiment** | Try 3 different chunk sizes (e.g., 200, 500, 1000) and compare retrieval quality — which size finds the best context for the same query? | 10 |
| **E. Hybrid retriever** | Combine similarity search with MMR or metadata filtering — explain when each strategy is better | 10 |
| **F. Streaming UI** | Build a simple interactive cell where users type questions and see the RAG chain stream its answer token by token | 10 |

---

### 🏗️ Starter Scaffold

The cells below provide a **skeleton structure** for your project. Each cell has `TODO` comments where you need to fill in your code. The structure follows the same 4-phase approach from the course.

> **Tip:** Refer back to the demo cells above (Cells 1–22) whenever you get stuck. The patterns are the same — you're just applying them to your own data now.

---

#### Task 1 — Load Your Documents

In [1]:
import shutil, os
from openai import OpenAI
from dotenv import load_dotenv
import time
from typing import List
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from openai import AzureOpenAI
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT    = "https://group2-bobeheus2.services.ai.azure.com/openai"
AZURE_OPENAI_API_VERSION = "2025-01-01-preview"
AZURE_OPENAI_API_KEY     = os.getenv("AZURE_OPENAI_API_KEY")
CHAT_DEPLOYMENT          = "o4-mini"
EMBEDDING_DEPLOYMENT     = "text-embedding-3-small"

azure_client = OpenAI(
    base_url=f"{AZURE_OPENAI_ENDPOINT}/openai/v1",
    api_key=AZURE_OPENAI_API_KEY
)

def get_embedding(text: str) -> List[float]:
    response = azure_client.embeddings.create(
        input=text,
        model=EMBEDDING_DEPLOYMENT
    )
    return response.data[0].embedding

print("Endpoint:", AZURE_OPENAI_ENDPOINT)
print("Embedding deployment:", EMBEDDING_DEPLOYMENT)
print("API version:", AZURE_OPENAI_API_VERSION)

d:\Users\bsi80267\.conda\envs\All-Rounder\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


Endpoint: https://group2-bobeheus2.services.ai.azure.com/openai
Embedding deployment: text-embedding-3-small
API version: 2025-01-01-preview


In [2]:
print("Endpoint:", repr(AZURE_OPENAI_ENDPOINT))
print("Embedding deployment:", repr(EMBEDDING_DEPLOYMENT))
print("API version:", repr(AZURE_OPENAI_API_VERSION))
print("API key (first 8 chars):", AZURE_OPENAI_API_KEY[:8] if AZURE_OPENAI_API_KEY else "NOT SET")

Endpoint: 'https://group2-bobeheus2.services.ai.azure.com/openai'
Embedding deployment: 'text-embedding-3-small'
API version: '2025-01-01-preview'
API key (first 8 chars): DQNPlPme


In [3]:
test_client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
)

# Print the exact URL that will be called
url = f"{AZURE_OPENAI_ENDPOINT}/deployments/{EMBEDDING_DEPLOYMENT}/embeddings?api-version={AZURE_OPENAI_API_VERSION}"
print("Full URL being called:", url)

Full URL being called: https://group2-bobeheus2.services.ai.azure.com/openai/deployments/text-embedding-3-small/embeddings?api-version=2025-01-01-preview


In [4]:
import requests

url = f"{AZURE_OPENAI_ENDPOINT}/deployments/{EMBEDDING_DEPLOYMENT}/embeddings?api-version={AZURE_OPENAI_API_VERSION}"

headers = {
    "Content-Type": "application/json",
    "api-key": AZURE_OPENAI_API_KEY,
}

body = {"input": "test", "model": EMBEDDING_DEPLOYMENT}

response = requests.post(url, headers=headers, json=body)
print("Status:", response.status_code)
print("Response:", response.json())

Status: 200
Response: {'object': 'list', 'data': [{'object': 'embedding', 'embedding': [-0.00986480712890625, 0.0015592575073242188, 0.0156707763671875, -0.0548095703125, -0.00640869140625, -0.012908935546875, 0.00966644287109375, -0.01355743408203125, 0.0286712646484375, 0.007843017578125, 0.03179931640625, -0.0065765380859375, 0.0038814544677734375, 0.01012420654296875, 0.0140533447265625, 0.044677734375, -0.059783935546875, -0.002445220947265625, -0.0511474609375, 0.036529541015625, 0.036163330078125, 0.02435302734375, 0.03997802734375, -0.043609619140625, 0.034454345703125, -0.0195770263671875, -0.01401519775390625, 0.0106353759765625, 0.03118896484375, -0.041534423828125, 0.05877685546875, -0.02886962890625, -0.0013904571533203125, -0.038818359375, 0.055206298828125, 0.004138946533203125, 0.0236053466796875, 0.018890380859375, -0.0056915283203125, -0.003925323486328125, -0.041473388671875, -0.044921875, 0.0113677978515625, 0.0272064208984375, 0.0299835205078125, -0.0186767578125, 

In [5]:
test_client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
)

response = test_client.embeddings.create(
    input="test",
    model=EMBEDDING_DEPLOYMENT
)
print("Success! Embedding length:", len(response.data[0].embedding))

NotFoundError: Error code: 404 - {'error': {'code': '404', 'message': 'Resource not found'}}

In [ ]:
web_loader = WebBaseLoader([
    "https://en.wikipedia.org/wiki/The_Legend_of_Heroes:_Trails_of_Cold_Steel",
    "https://en.wikipedia.org/wiki/The_Legend_of_Heroes:_Trails_of_Cold_Steel_II",
    "https://en.wikipedia.org/wiki/The_Legend_of_Heroes:_Trails_of_Cold_Steel_III",
    "https://en.wikipedia.org/wiki/The_Legend_of_Heroes:_Trails_of_Cold_Steel_IV"
])
all_docs = web_loader.load()

print(f"Web documents loaded: {len(all_docs)}")
print(f"Total: {len(all_docs)}")

#### Task 2 — Chunk Your Documents

Choose your `chunk_size` and `chunk_overlap` and **explain why** in a comment.

In [ ]:
# chunk size 500 dan chunk overlap 50 cukup untuk memberi keseimbangan antara konteks
# dengan efisiensi biaya

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(all_docs)

print(f"Panjang dokumen: {len(all_docs)}")
print(f"Total chunks: {len(chunks)}")

#### Task 3 — Store in ChromaDB

In [ ]:
PERSIST_DIR = "./cold_steel_db"
if os.path.exists(PERSIST_DIR):
    import gc
    gc.collect()
    
    for _ in range(5):
        try:
            shutil.rmtree(PERSIST_DIR)
            break
        except OSError as e:
            print(f"Error removing directory: {e}. Retrying...")
            time.sleep(0.5)
    if os.path.exists(PERSIST_DIR):
        raise Exception(f"Failed to remove {PERSIST_DIR} after multiple attempts.")


embeddings = AzureOpenAIEmbeddings(
    azure_deployment=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
    collection_name="cold_steel_lore",
)

print(f"Stored {len(chunks)} chunks")

#### Task 4 — Build & Test the Retriever

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

test_queries = [
    "Who is Rean Schwarzer?",
    "What is the setting of the Erebonian Empire?",
    "How does the combat system work?",
]

for query in test_queries:
    docs = retriever.invoke(query)
    print(f"\n🔎 Query: \"{query}\"")
    for i, doc in enumerate(docs):
        print(f"   [{i+1}] {doc.page_content[:100]}...")

#### Task 5 — Design Your Custom RAG Prompt

Create a **custom system message** that shapes how the model answers. Be creative and specific.

In [ ]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are an expert on 'The Legend of Heroes: Trails of Cold Steel' series.
    Answer the question using the provided context.
    - Cite your sources clearly using the URL from the metadata.
    - If you don't know the answer, state that the information is not in the records.
    - Use bullet points for characters or gameplay mechanics.
    """),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

#### Task 6 — Assemble & Test the Full RAG Chain

In [ ]:
model = AzureChatOpenAI(
    azure_deployment=AZURE_OPENAI_CHAT_DEPLOYMENT,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0,
)


def format_docs(docs):
    return "\n\n".join(f"[Source: {d.metadata.get('source')}] {d.page_content}" for d in docs)

rag_chain = (
    RunnableParallel(
        context = retriever | format_docs,
        question = RunnablePassthrough(),
        chat_history = lambda x: x.get("chat_history", [])
    )
    | rag_prompt
    | model
    | StrOutputParser()
)

questions = [
    "What is Class VII?",
    "Who are the main protagonists?",
    "Explain the setting of the game.",
    "What are Tactical Link systems?",
    "Which empire does the story take place in?"
]

for q in questions:
    print(f"\n❓ {q}\n💡 {rag_chain.invoke({'question': q, 'chat_history': []})}\n" + "-"*30)

#### Task 7 — Bonus Challenge(s)

Pick **at least one** bonus challenge from the table above and implement it below.

In [ ]:
print("--- Challenge A: Multi-turn Memory ---")
chat_history = []
q1 = "Who is the protagonist of Trails of Cold Steel?"
a1 = rag_chain.invoke({"question": q1, "chat_history": chat_history})
chat_history.extend([HumanMessage(content=q1), AIMessage(content=a1)])
print(f"Q: {q1}\nA: {a1[:100]}...")

q2 = "What is his special weapon?"
a2 = rag_chain.invoke({"question": q2, "chat_history": chat_history})
chat_history.extend([HumanMessage(content=q2), AIMessage(content=a2)])
print(f"\nFollow-up Q: {q2}\nA: {a2[:100]}...")

q3 = "Which shool does he attend?"
a3 = rag_chain,invoke({"question": q3, "chat_history": chat_history})
print(f"\nFollow-up Q: {q3}\nA: {a3[:100]}...")

In [ ]:
print("--- Challenge B: Source Citation (with document metadata)")

def format_docs_with_citation(docs):
    parts = []

    for d in docs:
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", "N/A")
        parts.append(f"[Source: {src}] | Page: {page}]\n{d.page_content}")
    return "\n\n".join(parts)

rag_chain_with_sources = RunnableParallel(
    answer = (
        RunnableParallel(
            context = retriever | format_docs_with_citation,
            question = RunnablePassthrough(),
            chat_history = lambda x: x.get("chat_history", [])
        )
        | rag_prompt
        | model
        | StrOutputParser()
    ),
    source_docs = retriever
)

cite_query = "What is Class VII and who are its members?"
result = rag_chain_with_sources.invoke({"question": cite_query, "chat_history": []})

print(f"\n{cite_query}")
print(f"\nAnswer:\n{result['answer']}")
print("\nSources used:")

seen = set()
for doc in result["source_docs"]:
    src = doc.metadata.get("source", "unknown")
    page = doc.metadata.get("page", "N/A")
    key = (src, page)
    if key not in seen:
        print(f" - {src} (page {page})")
        seen.add(key)


In [ ]:
print("--- Challenge C: Evaluation Harness (LLM-as-Judge)")

ground_truth = [
    {"question": "What is Rean Schwarzer's ogre power?",
     "expected": "Rean possesses the ability to tap into an Ogre/Awakener power that dramatically boosts his combat strength."},
    {"question": "Which military academy do the characters attend?",
     "expected": "They attend Thors Military Academy."},
    {"question": "What is the name of Rean's signature sword style?",
     "expected": "Rean practices the Eight Leaves One Blade sword style."},
    {"question": "Who is the antagonist in Trails of Cold Steel III?",
     "expected": "Osborne and the Ouroboros society are the primary antagonists, with Ash Carbide as a key figure."},
    {"question": "What country is the Trails of Cold Steel series set in?",
     "expected": "The games are set in the Erebonian Empire."},
]

judge_llm = model

eval_results = []
for item in ground_truth:
    rag_answer = rag_chain.invoke({"question": item["question"], "chat_history": []})

    judge_prompt = (
        f"Expected key fact: {item['expected']}\n"
        f"RAG answer: {rag_answer}\n\n"
        "Does the RAG answer cover the expected key fact? "
        "Reply with exactly one word: Yes or No."
    )

    verdict_msg = judge_llm.invoke(judge_prompt)
    verdict = verdict_msg.content.strip().split()[0]
    eval_results.append({"question": item['question'], "verdict": verdict})
    print(f"{'Good' if verdict.lower() == 'yes' else 'Bad'} [{verdict}] {item['question']}")

passed = sum(1 for r in eval_results if r["verdict"].lower() == "yes")
print(f'\nScore: {passed}/{len(ground_truth)} questions passed')

In [ ]:
print("--- Challenge D: Chunking Experiment (200/500/1000)")

chunk_experiment_query = "Explain the Tactical Link combat system."
chunk_sizes = [200, 500, 1000]

for size in chunk_sizes:
    exp_dir = f"./exp_chroma_{size}"
    if os.path.exists(exp_dir):
        shutil.rmtree(exp_dir)

    exp_splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=max(20, size // 10),
    )
    exp_chunks = exp_splitter.split_documents(all_docs)

    exp_store = Chroma.from_documents(
        documents=exp_chunks,
        embedding=embeddings,
        persist_directory=exp_dir,
        collection_name=f"cold_steel_{size}",
    )
    exp_retriever = exp_store.as_retriever(search_kwargs={"k": 3})
    retrieved = exp_retriever.invoke(chunk_experiment_query)

    total_chars = sum(len(d.page_content) for d in retrieved)
    print(f"\nchunk_size={size} | chunks_total={len(exp_chunks)} | retrieved_chars={total_chars}")
    for j, doc in enumerate(retrieved):
        preview = doc.page_content[:120].replace("\n", " ")
        print(f"   [{j+1}] {preview}...")
e
    shutil.rmtree(exp_dir)

print("""
Observation:
  • size=200  — very granular; retrieves precise sentences but may miss
                surrounding context needed to fully explain a mechanic.
  • size=500  — balanced; each chunk covers a coherent paragraph,
                making it the best trade-off for dense JRPG lore.
  • size=1000 — broad context but fewer chunks fit in the k-budget,
                risking dilution if a single chunk mixes multiple topics.
→ Conclusion: chunk_size=500 yields the most informative retrieved context
  for gameplay-mechanic questions in this corpus.
""")


In [ ]:
print("\n--- Challenge E: Hybrid Retriever (MMR + metadata filter) ---")

mmr_query = "What are the main story events of Trails of Cold Steel?"

sim_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)
sim_docs = sim_retriever.invoke(mmr_query)
print("\n🔵 Similarity — top-4 sources:")
for d in sim_docs:
    print(f"   • {d.metadata.get('source', 'N/A')[:80]}")

mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.5},
)
mmr_docs = mmr_retriever.invoke(mmr_query)
print("\n🟢 MMR (lambda=0.5, fetch_k=20) — top-4 sources:")
for d in mmr_docs:
    print(f"   • {d.metadata.get('source', 'N/A')[:80]}")

cs1_url = "https://en.wikipedia.org/wiki/The_Legend_of_Heroes:_Trails_of_Cold_Steel"
filtered_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {"source": cs1_url},
    },
)
filtered_docs = filtered_retriever.invoke(mmr_query)
print(f"\nMetadata filter (source = Cold Steel I only) — {len(filtered_docs)} docs returned:")
for d in filtered_docs:
    print(f"   • {d.metadata.get('source', 'N/A')[:80]}")

print("""
When to use each strategy:
  • Similarity  — best for focused factual questions where you want
                  the single most relevant passage (e.g. 'What year
                  was Rean born?').
  • MMR         — best for broad / summarisation queries where you
                  want variety across different games or plot arcs
                  rather than 4 nearly-identical paragraphs.
  • Filter      — best when the user explicitly restricts scope
                  (e.g. 'Only tell me about Cold Steel I') or when
                  you want to avoid cross-game spoilers.
""")

In [ ]:
print("\n--- Challenge F: Streaming UI ---")
query = "Summarize the plot of the first game."
print(f"Streaming response for: {query}\n")
for chunk in rag_chain.stream({"question": query, "chat_history": []}):
    print(chunk, end="", flush=True)
    time.sleep(0.01)

---

### 📦 Submission Checklist

Before submitting, verify the following:

- [ ] **Task 1:** Loaded 3+ documents from 2+ different source types
- [ ] **Task 2:** Chunking strategy implemented with a written justification comment
- [ ] **Task 3:** Chroma vector store created with persistence to disk
- [ ] **Task 4:** Retriever tested with 3 queries, showing top-K results with metadata
- [ ] **Task 5:** Custom RAG prompt with a specific, thoughtful system message
- [ ] **Task 6:** Full RAG chain assembled with LCEL pipes, tested with 5+ questions
- [ ] **Task 7:** At least one bonus challenge completed
- [ ] **All cells run** top-to-bottom without errors (Kernel → Restart & Run All)
- [ ] **No hardcoded API keys** — uses environment variables or `.env` file

### 📁 What to Submit

1. This notebook (`.ipynb`) with all cells executed and outputs visible
2. Your `.env.example` file (with placeholder values, NOT real keys)
3. A short `README.md` (3–5 sentences) describing:
   - What documents you chose and why
   - What bonus challenge(s) you completed
   - One thing you learned or found surprising

---

### 💡 Tips for Success

- **Start with small, simple documents** — don't load 500 pages on your first try.
- **Test each task independently** before chaining them together.
- **Print intermediate results** — check what your retriever returns before building the full chain.
- **Experiment with chunk sizes** — this is the single biggest lever for RAG quality.
- **Read the error messages** — LangChain errors are usually descriptive and tell you exactly what's wrong.

Good luck! 🚀

---

## 🧹 Cleanup

In [ ]:
# ============================================================
# CLEANUP: Remove persisted Chroma database
# ============================================================
import shutil, os

if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")
    print("🗑️  Removed './chroma_db/' directory")
else:
    print("ℹ️  Nothing to clean up")

print("\n✅ Demo complete! You've built a full RAG pipeline with Azure OpenAI + ChromaDB + LangChain.")